In [1]:
import pandas as pd
from math import ceil
import json

Loads the random sample

In [7]:
data = pd.read_csv('data_rs.csv')

Keep the needed fields

In [15]:
chosen = data[['avisoid', 'avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo']]
chosen['avisoid'] = chosen['avisoid'].astype(int)

/tmp/ipykernel_11535/926371173.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chosen['avisoid'] = chosen['avisoid'].astype(int)


Assign a group to each ad

In [16]:
values = pd.Series(range(1, 6))

chosen['group'] = pd.concat([values] * (40 // len(values)), ignore_index=True)

/tmp/ipykernel_11535/3074604153.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chosen['group'] = pd.concat([values] * (40 // len(values)), ignore_index=True)


Here we start preparing the JSON resource for the app, the key is the 'avisoid' variable and the value of the item is a list  in this order of the other variables ['avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo']

In [30]:
chosen['list'] = chosen.apply(lambda row: [row[col] for col in ['avisocargo', 'avisocuerpo', 'disponibilidadnombre',
       'avisorequisitos', 'avisolugartrabajo']], axis=1)

/tmp/ipykernel_11535/1640821379.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chosen['list'] = chosen.apply(lambda row: [row[col] for col in ['avisocargo', 'avisocuerpo', 'disponibilidadnombre',


In [32]:
json_data = chosen.set_index('avisoid')['list'].to_dict()

Creamos los grupos de avisos en funcion del JSON generado

In [33]:
# Número de grupos deseados
num_groups = 5

# Calcular la longitud de cada grupo
group_size = ceil(len(json_data) / num_groups)

In [34]:
grouped_dicts = {}
for i, key in enumerate(json_data.keys()):
    group_index = i // group_size
    if group_index not in grouped_dicts:
        grouped_dicts[group_index] = {}
    grouped_dicts[group_index][key] = json_data[key]

In [38]:
with open('jobads2.json', 'w') as json_file:
    json.dump(grouped_dicts, json_file, indent=4)